# Day 2 Homework: Ollama webpage summarizer

This upgrades the Day 1 webpage summarizer so the summary is generated by a local open-source model through Ollama instead of the OpenAI API.

Ollama runs open-source language models on this machine and exposes a local HTTP server. The server normally listens at `http://localhost:11434`, so requests stay on the local computer.

The `/v1` path gives Ollama an OpenAI-compatible endpoint. That means the OpenAI Python client can be pointed at Ollama with `base_url="http://localhost:11434/v1"`, while `api_key="ollama"` is only a placeholder expected by the client library. No OpenAI API key is needed for this local summary.

The Day 1 implementation is kept as close as possible to the course version:

- `fetch_website_contents(...)` is still imported from the existing Week 1 scraper.
- The same system prompt and user prompt are preserved.
- `messages_for(...)`, `summarize(...)`, and `display_summary(...)` keep the same roles.
- The OpenAI-hosted model call is replaced with an Ollama call to `llama3.2`.

Benefits of this local approach: no API charges, and the scraped website text stays on the local machine.

In [ ]:
import sys
from pathlib import Path

from IPython.display import Markdown, display
from openai import OpenAI

week1_path = next(
    path / "week1"
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "week1" / "scraper.py").exists()
)
if str(week1_path) not in sys.path:
    sys.path.insert(0, str(week1_path))

from scraper import fetch_website_contents

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2"

ollama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"
)

In [ ]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [ ]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [ ]:
def summarize(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=messages_for(website)
    )
    return response.choices[0].message.content

In [ ]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

Run the main homework test with the required default model, `llama3.2`.

In [ ]:
display_summary("https://edwarddonner.com")

Optional variation: if `gemma3` is installed in Ollama, the same application can use it by changing only the `MODEL` value.

In [ ]:
# Optional model swap. Leave llama3.2 as the default homework model above.
# MODEL = "gemma3"
# display_summary("https://edwarddonner.com")